# Task 3 — Gender and Usage Classification

**Status:** evidence-led implementation scaffold. The data, baseline, metric, and
experiment contracts are fixed below. No model has been trained and no result is claimed.

**Owner:** TODO(owner)


## Colab setup — repository and teacher data

Run these cells at the start of each fresh Colab runtime. They are safe to rerun.
The repository supplies `data/processed`, including the one allowed split. The Drive
archive supplies only `data/raw/teacher`. Images are extracted to Colab's local disk
because training directly from Drive is much slower.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=cwd, check=True)


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("This setup cell must run in Google Colab.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        print("Local repository changes found; skipped the automatic fast-forward update.")
    else:
        run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR]
    )

print(f"Repository ready: {REPO_DIR} ({BRANCH})")


In [ ]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(
        f"Dataset archive not found at {DATA_ZIP}. Check the Drive folder and file name."
    )

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [
        name for name in names if Path(name).is_absolute() or ".." in Path(name).parts
    ]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and name.lower().endswith((".jpg", ".jpeg"))
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive does not contain teacher images in the expected layout.")

    current_images = sum(1 for path in teacher_dir.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg"})
    needs_extract = current_images != expected_images or not all(path.is_file() for path in required_files)
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images from Drive...")
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(1 for path in teacher_dir.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg"})
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )

print(f"Teacher data ready: {actual_images:,} images in {teacher_dir}")


In [ ]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (
    DRIVE_TASK_DIR / "checkpoints",
    DRIVE_TASK_DIR / "logs",
    DRIVE_TASK_DIR / "results",
):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.data import get_samples, load_label_maps, load_splits

splits = load_splits()
label_maps = load_label_maps()
gender_development = get_samples(splits, partition="development", target="gender")
usage_development = get_samples(splits, partition="development", target="usage")

print("Task 3 data is ready.")
print(f"Gender development rows: {len(gender_development):,}")
print(f"Usage development rows:  {len(usage_development):,}")
print(f"Persistent outputs:       {DRIVE_TASK_DIR}")
splits.groupby("partition", observed=True).size().rename("rows").to_frame()


## 1. Task contract

- Decision: suggest the supplied catalogue target-audience and usage labels from one teacher image.
- Prediction unit: one teacher catalogue image and product ID.
- Outputs: separate `gender` and `usage` labels. The merged official file remains
  `id,gender,articleType,season,usage`.
- In scope: catalogue-assistance suggestions, confidence review, and assignment predictions.
- Out of scope: person identity or gender inference, customer profiling, access, pricing, or
  another high-impact decision.
- Owner and approval date: TODO(owner).


## 2. EDA evidence and method roles

The primary baseline is derived from the saved Notebook 01 evidence, not from a model-name list.
The evidence fixes the following boundaries:

- 32,773 development products form 22,905 family-safe groups, so all model evidence uses the
  five saved family folds rather than a new random split;
- almost all images are 60×80 and only 12 have unusual dimensions, so the baseline uses the
  native `(height=80, width=60)` information instead of inventing detail by upscaling;
- the contact sheet shows one main product at varying sizes and positions, so shared local
  filters and a position-light whole-image summary are appropriate;
- small visible cues include straps, collars, sleeves, shoe outlines, and fabric boundaries, so
  the feature extractor must preserve several spatial regions instead of reducing immediately
  to one global vector;
- 294 grayscale images remain valid and sampled brightness varies strongly, so every image
  converts to RGB and each fold fits its own content-pixel mean and standard deviation;
- the transform-risk figure shows that stretch changes shape and centre crop can remove edge
  content, so only unusual images use aspect-preserving letterbox resize;
- gender ranges from 17,753 examples in its largest class to 543 in its smallest; usage ranges
  from 25,151 `Casual` examples to one `Home` example, so the first loss stays unweighted and
  macro-F1 exposes minority collapse;
- one usage value is truly blank, literal `NA` is a real class, and only 26 of 45 possible
  gender–usage pairs occur, so the two targets start as separate masked models;
- article type agrees with about 78.5% of gender and 90.0% of usage labels under a descriptive
  majority lookup. It is therefore an error slice and shortcut warning, never a model input.

| Role | Method | Purpose |
|---|---|---|
| Lower bounds | Fold-training majority and stratified predictors | Check priors, metrics, masks, and seeds |
| Integrity checks | Bias-only, shuffled-label, tiny-batch overfit | Find broken code or leakage |
| Classical reference | Exact HOG + HSV logistic regression | Measure hand-built shape and colour |
| Primary learnable baseline | Exact small scratch CNN, trained separately per target | Start the parent-child chain |


## 3. Data, masks, and five-fold contract

- Sole split: `data/processed/splits.csv`. Do not create another split.
- CV mode: all five saved family-safe folds.
- OOF rule: every valid development image is predicted once by a model that did not train on
  its fold or family.
- Teacher images only. Optional Task 4 images never enter Task 3.
- Label-mask contract: gender uses `has_gender_label`; usage uses `has_usage_label`.
- The one blank usage row contributes no usage loss or metric. Literal `NA` is a real class.
- Keep all five gender and nine usage logits in every fold, including `Home`.
- Holdout and quarantine targets remain sealed until Notebook 06 after method freeze.
- Product-family grouping is the leakage and bootstrap unit.


## 4. Baseline preprocessing and rationale

| Choice | Exact baseline value | Reason |
|---|---|---|
| Input | `3×80×60` | Matches native image information |
| Geometry | EXIF, RGB, aspect preservation, LANCZOS letterbox only for unusual images | Avoid stretch and crop damage |
| Padding | White before scaling; neutral zero after standardisation | Match catalogue boundary without fitting padding |
| Normalisation | RGB mean/std fitted on current fold-training content pixels | Keep learned preprocessing inside the fold |
| Augmentation | None | Expose the raw train–validation and robustness gap |
| Validation | Deterministic transform only | Make candidate comparisons exact |

`(128,96)`, augmentation, another loss, or another transform is not a mandatory menu. It is a
single-factor child only when an accepted parent's evidence triggers it.


## 5. Lower bounds and classical reference

Lower bounds derive their majority or class prior from each training complement.

Exact classical reference:

- grayscale HOG: 9 orientations, `(8,8)` pixels per cell, `(2,2)` cells per block, `L2-Hys`;
- HSV colour: 16 bins per channel;
- `StandardScaler` fitted inside the training complement;
- multinomial logistic regression: L2, `C=1.0`, `lbfgs`, `max_iter=2000`, no class weights;
- separate gender and usage fits; restore absent fixed-class output columns as zero probability.

The ground-truth article-type lookup stays a shortcut diagnostic. It is not a baseline model.


## 6. Exact primary learnable baseline

Train one fixed design as two separate scratch models:

```text
Conv3x3 3->32   + BatchNorm + ReLU + MaxPool2
Conv3x3 32->64  + BatchNorm + ReLU + MaxPool2
Conv3x3 64->128 + BatchNorm + ReLU + MaxPool2
Conv3x3 128->256 + BatchNorm + ReLU
Adaptive global average pooling
Linear 256->5 or Linear 256->9
```

### Why this depth and spatial reduction follow from the EDA

The three pooling steps follow the observed native image size:

```text
80×60 -> 40×30 -> 20×15 -> 10×7
```

A `10×7` final grid still keeps separate evidence for small product regions. A fourth pool would
reduce these tiny inputs to about `5×3`, making it easier to lose straps, sleeves, collars, and
other edge details seen in the contact sheet. The fourth convolution therefore adds feature
capacity without another spatial reduction. With four `3×3` convolutions and three pools, one
final-grid position sees about a `38×38` source region; global average pooling then combines the
whole set of positions without tying a feature to one fixed product location.

### Honest boundary on the channel counts

The EDA justifies a small native-resolution CNN, three reductions, local-to-global feature
learning, and a bounded scratch capacity. It does **not** prove that `32,64,128,256` is optimal.
Those widths are a predeclared capacity hypothesis. As each pool removes about three quarters
of the spatial positions, the next block gets more feature channels while total activation size
continues to fall. The result is 390,181 trainable parameters for gender and 391,209 for usage.
The first curves decide whether this hypothesis was too small, too large, or trained for too
short a time; the EDA alone does not make that claim.

Batch normalization and ReLU are optimisation controls, not EDA discoveries. No dropout, class
weighting, augmentation, or pretrained weights are added because there is no parent-model
evidence for those changes yet.

Fixed recipe: Kaiming random initialisation, unweighted cross-entropy, AdamW, learning rate
`0.001`, weight decay `0.0001`, batch 128, 30 epochs, cosine decay to `0.00001`, no mixed
precision, no early stopping, final-epoch OOF checkpoint, seed 2753.

If curves have not settled at epoch 30, the first child changes only the budget. Do not switch
architecture.


In [ ]:
from fashion.train.task3_baseline import check_task3_baseline_setup

BASELINE_TARGET = "gender"  # Run and diagnose one target before changing this.
baseline_check = check_task3_baseline_setup(
    BASELINE_TARGET,
    root=REPO_DIR,
    device_name="cuda",
)
baseline_check


In [ ]:
from fashion.train.task3_baseline import run_task3_baseline_cv

START_BASELINE_TRAINING = False

if START_BASELINE_TRAINING:
    baseline_result = run_task3_baseline_cv(
        BASELINE_TARGET,
        root=REPO_DIR,
        output_root=DRIVE_TASK_DIR,
        registry_path=REPO_DIR / "results/runs.csv",
        registry_mirrors=[DRIVE_TASK_DIR / "results/runs.csv"],
        device_name="cuda",
    )
    baseline_result
else:
    print("Baseline check passed. Set START_BASELINE_TRAINING=True only when ready.")


## 7. Frozen evaluation and required diagnosis

- Primary development metric for `gender`: pooled five-fold OOF macro-F1 over all 5 classes.
- Primary development metric for `usage`: pooled five-fold OOF macro-F1 over all 9 classes.
- Mandatory usage companion: macro-F1 without `Home`.
- Secondary: accuracy, balanced accuracy, weighted-F1, MCC, NLL, Brier, ECE, and joint exact match.

Before another configuration is written, every E1 candidate must have:

1. train/validation loss and macro-F1 curves;
2. pooled OOF and fold metrics;
3. per-class support, predicted count, precision, recall, F1, and confusion matrices;
4. fixed success, failure, high-confidence-error, rare, grayscale, and unusual-size examples;
5. parameters, bytes, training time, peak memory, and named-device latency;
6. JPEG 75, brightness ×0.85/×1.15, 3% translation, and grayscale robustness;
7. a written accept, reject, or stop decision.


## 8. Gender baseline diagnosis and next gate

**Status: WAITING FOR REGISTERED BASELINE EVIDENCE.**

| Observed accepted-parent weakness | Only eligible next factor |
|---|---|
| Training still improves at epoch 30 | Epoch or schedule budget |
| Train and validation are both weak, close, and settled | Low-resolution-stem ResNet-18 architecture |
| Strong train score but weak OOF | Fixed light augmentation |
| Boys, Girls, or Unisex collapses while common classes work | Capped class-balanced CE |
| Fixed failures show lost detail without overfit | Input `(128,96)` |
| Quality passes but named-device cost fails | Scratch MobileNetV3-Small architecture |
| Colour/brightness robustness fails | Matching mild colour augmentation |

Choose one row only. ResNet and MobileNet are never run together as a screen.


## 9. Usage baseline diagnosis and next gate

**Status: WAITING FOR REGISTERED BASELINE EVIDENCE.**

| Observed accepted-parent weakness | Only eligible next factor |
|---|---|
| Training still improves at epoch 30 | Epoch or schedule budget |
| Casual dominates while supported minorities collapse | Capped class-balanced CE |
| Common classes are weak and train/validation are close | Low-resolution-stem ResNet-18 architecture |
| Strong train score but weak OOF | Fixed light augmentation |
| Fixed failures show lost detail without overfit | Input `(128,96)` |
| Quality passes but named-device cost fails | Scratch MobileNetV3-Small architecture |
| Errors are mainly weak business labels or 1–22 examples | Stop and record the data limit |

`Home` never selects a child. Class-balanced CE, if triggered, uses effective-number weights
with `beta=0.999`, mean-one nonzero normalisation, cap 5, and fold-training counts only.


## 10. Parent-child hypothesis card

Create this record before any child configuration or run:

```text
hypothesis_id: TODO(after parent diagnosis)
target: TODO
parent_experiment_id: TODO
parent_run_ids: TODO
observed_weakness: TODO
trigger_observation_ids: TODO
evidence_paths: TODO
hypothesis: TODO
single_changed_factor: TODO
fixed_controls: TODO
expected_result: TODO
rejection_condition: TODO
created_before_child_run: true
```

No child may change architecture and input, loss, augmentation, optimiser, or budget together.


## 11. Sequential experiment and decision ledger

This table grows one approved row at a time. Do not fill a model matrix in advance.

| Hypothesis | Target | Parent run IDs | One changed factor | Child run IDs | Evidence bundle | Decision | Accepted parent |
|---|---|---|---|---|---|---|---|
| Baseline contract | gender | none | primary small CNN | TODO(after execution) | TODO | TODO | TODO |
| Baseline contract | usage | none | primary small CNN | TODO(after execution) | TODO | TODO | TODO |

Decision values are `accept`, `reject`, or `stop`. A rejected child's change is not stacked
into the next run. Every number shown here must be generated from registered artifacts.


## 12. Conditional sharing and pretrained comparison

A shared two-head model is not automatic. Open it only when accepted separate parents exist,
their combined measured cost fails a named limit, a matched common backbone is possible, and the
one-point no-harm margin is frozen first. Compare negative transfer separately for gender and
usage. Sharing changes only the task structure.

One ImageNet-pretrained ResNet-18 benchmark may run only after eligible selection is fixed. Mark
`scratch=false` and every submission, official-prediction, and application eligibility field
false. It cannot become a parent or change the eligible winner.


## 13. Implementation order and run registry

The reusable baseline path is now implemented in `src/fashion/train/`:

1. `registry.py` appends one durable row before the first optimiser step and preserves failures;
2. `data.py` enforces the saved folds, target mask, fold-only RGB statistics, and corruptions;
3. `model.py` contains only the exact scratch CNN baseline;
4. `metrics.py` keeps the fixed class order, including zero-support classes;
5. `task3_baseline.py` writes curves, OOF predictions, per-class metrics, confusion, failures,
   core robustness, checkpoint, runtime, and cost artifacts.

The first launch cell is check-only and must report `optimizer_steps: 0`. The second launch cell
keeps `START_BASELINE_TRAINING=False` until the owner deliberately changes it. The five-fold
runner rejects partial E1 fold lists. Lower bounds and the classical reference remain separate
comparison stages and do not alter this primary CNN configuration.

Every training execution appends through `fashion.train.registry` to repository
`results/runs.csv` before its first optimiser step. Drive folders may mirror large artifacts,
but they do not replace the canonical registry. Notebook cells call shared code; they do not
contain a second training implementation.


## 14. Finalist judgement and decision log

Confirm only the accepted final parent for each target, plus a shared candidate only if its cost
gate opened and it passed. Use all five folds and three fixed seeds with every method choice frozen.

| Final decision | Accepted chain | Rejected children and reasons | Seed evidence | Class/failure evidence | Robustness/cost | Limitation | Date/owner |
|---|---|---|---|---|---|---|---|
| Gender | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| Usage | TODO | TODO | TODO | TODO | TODO | TODO | TODO |
| Separate or shared system | TODO | TODO | TODO | TODO | TODO | TODO | TODO |

Do not rewrite earlier decisions after holdout access.


## 15. Handoff to final evaluation

Before Notebook 06, provide:

- accepted parent-child chain and rejected hypotheses: TODO(owner)
- frozen winning eligible run ID(s): TODO(owner)
- frozen preprocessing, model, loss, optimiser, schedule, and seed rules: TODO(owner)
- frozen metrics, thresholds, and five-fold/three-seed evidence: TODO(owner)
- final all-development scratch-refit procedure: TODO(owner)
- expected checkpoint, calibration, registry, and official-output paths: TODO(owner)
- shared-model gate result or recorded reason it was not opened: TODO(owner)
- unresolved risks, weak classes, and honest limitations: TODO(owner)

**Handoff status: NOT READY — owner must complete every item above.**
